# [BLOCK-T374 Refrigeration Operations at different elevation and rotation angles]

The description in the test step says:

> The block will move the telescope to Elevation Angle and will move the rotator to:
> -90º, -60º, -30º, 0º, 30º, 60º, 90º  
>   
> The block will sleep for 5 min on every rotator position.  
> Here is the configuration you will need:  
> 
> id: BLOCK-T374  
> override:  
>   elevation_angle: {elevation_angle}

[BLOCK-T374 Refrigeration Operations at different elevation and rotation angles]: https://rubinobs.atlassian.net/projects/BLOCK?selectedItem=com.atlassian.plugins.atlassian-connect-plugin:com.kanoah.test-manager__main-project-page#!/v2/testCase/228982672 

In [ ]:
import numpy as np
import os

from lsst.ts.block.utils import build_configuration_schema
from lsst.ts.observing import ObservingBlock, ObservingScript

In [ ]:
name = "BLOCK-T374"
program = "BLOCK-T374"
constraints = []
scripts = []

output_folder = "output_blocks"

In [ ]:
properties = {
    "azimuth": {
        "description": "Target azimuth angle",
        "type": "number",
        "default": 0,
    },
    "elevation": {
        "description": "Target elevation angle",
        "type": "number",
        "default": 80,
    },
    "ignore": {
        "description": "Name of the CSCs we want to ignore",
        "type": "array",
        "default": ["mtaos", "mtdome", "mtdometrajectory"],
    },
}

block_number = name.split("-")[-1]
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

In [ ]:
move_p2p = ObservingScript(
    name="maintel/move_p2p.py",
    standard=True,
    parameters=dict(
        az="$azimuth",
        el="$elevation",
        move_timeout=2700,
        ignore="$ignore",
    ),
)

sleep = ObservingScript(
    name="sleep.py", 
    standard=True, 
    parameters=dict(sleep_for=5 * 60.)
)

rotator_angles = [
    -89.5, 
    -60,
    -30, 
    0,
    30,
    60,
    89.5,
]

scripts = [move_p2p]
for rot in rotator_angles:
    move_rotator = ObservingScript(
        name="mtrotator/move_rotator.py", 
        standard=True,
        parameters=dict(angle=rot)
    )
    
    scripts.append(move_rotator)
    scripts.append(sleep)


move_rotator = ObservingScript(
    name="mtrotator/move_rotator.py", 
    standard=True,
    parameters=dict(angle=0)
)
scripts.append(move_rotator)

In [ ]:
block = ObservingBlock(
    name=name,
    program=program,
    scripts=scripts,
    constraints=constraints,
    configuration_schema=configuration_schema,
)

In [ ]:
block.model_dump_json(indent=2)

os.makedirs(output_folder, exist_ok=True)
output_path = f"{output_folder}/{name}.json"

with open(output_path, "w") as file:
    file.write(block.model_dump_json(indent=2))